# YOLO26m · EDA 기반 5-fold GPU 베이스라인

프로젝트의 **uv Python 3.12 가상환경을 커널로 선택**한 뒤 위에서 아래로 실행합니다.
데이터 준비 셀은 준비 파일을 만들고, **학습 셀은 실제로 5회 GPU 학습을 시작**합니다.
작성된 노트북에는 실행 결과를 저장하지 않았습니다.

| 항목 | 설정 |
|---|---|
| 모델 / 전이학습 | yolo26m.pt / full fine tuning (freeze=None) |
| 원본 | train 232장, test 842장. test는 학습·검증에서 제외 |
| 라벨 | 현재 파이프라인의 v1 보정 라벨. 원본 JSON을 수정하지 않음 |
| 분할 | setup30fix_v1_s42, StratifiedGroupKFold(5), seed=42 |
| 희귀 알약 | v1 전체 bbox가 5개 이하인 클래스가 든 **세팅 묶음 전체를 모든 fold의 train에 고정** |
| 비율 | 나머지 묶음을 약 8:2로 순환 검증. 전체 이미지를 185/47로 강제하지 않음 |
| epochs / patience | 최대 30 / mAP75–95가 5에폭 연속 개선되지 않으면 중단 (10으로 변경 가능) |
| optimizer / lr0 | MuSGD / 0.001 |
| batch / imgsz | 16 / 640 |
| 증강 | 기존 AUG_PRESETS["on"]: 회전 우선, 반전 끔, 약한 hue |
| 평가 / 그림 신뢰도 | mAP 평가 0.001 / 혼동행렬 0.001·0.25 / 이미지 표시 0.25 |
| 후처리 | 클래스별 NMS, IoU=0.5, max_det=4 |
| 성능 | mAP@[0.75:0.95], Precision, Recall, GPU 추론속도. 5-fold 평균·표준편차 |
| 저장 | fold별 best.pt, train/val 곡선, 클래스별 AP, 혼동행렬, 검증 GT·예측 비교 이미지 |

**검증에서 역전파나 optimizer.step()을 하지 않습니다.**
eval 모드·inference_mode를 적용하고 검증 전후 가중치·버퍼가 같은지도 검사합니다.

실행 시 산출물은 runs/ 아래 새 실험 폴더에 모읍니다. 기존 raw·보정 CSV·공용 산출물은
덮어쓰지 않습니다. YOLO 라벨·분할 목록·data.yaml은 data_process.data_main.run()이 만듭니다.

In [ ]:
import os
import sys
from pathlib import Path

# 루트 또는 notebooks에서 실행해도 프로젝트 import 경로를 찾는다.
PROJECT_PATH = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "config.py").is_file()
    ),
    None,
)
if PROJECT_PATH is None:
    raise RuntimeError("pill-bbox 프로젝트 안에서 노트북을 실행하세요.")
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_AUTOINSTALL"] = "false"

In [ ]:
import gc
import json
import shutil
from copy import copy
from datetime import datetime
from time import perf_counter
from unittest.mock import patch
from zoneinfo import ZoneInfo

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from IPython.display import Image, display
from matplotlib.patches import Rectangle
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.metrics import ConfusionMatrix
from ultralytics.utils.torch_utils import EarlyStopping, init_seeds

from data_process import data_yaml
from data_process.data_processing import list_images
from data_process.make_labels import LABEL_VERSIONS, file_sha256
from data_process.split import MIN_SAME, N_FOLDS, RARE_MAX, rare_classes
from models.evaluate import CONFUSION_PAIRS, MAIN_METRIC, ap75_95, map75_95, size_group
from models.train import AUG_PRESETS, MAIN_METRIC_KEY, Map7595Trainer
from src.config import PROJECT_ROOT, RUNS_DIR, get_device

## 1. 설정과 GPU

full fine tuning을 사용합니다. 그 밖의 미지정 학습 인자는 Ultralytics 기본값이며 실제 인자를
fold별 args.yaml에 남깁니다. lr0는 시작 학습률입니다. warmup·스케줄과 MuSGD의 일부
head 그룹 3배 배율은 유지하고 실제 그룹별 학습률을 기록합니다.

설치된 Ultralytics의 nms=True 경로를 명시해 NMS IoU=0.5가 실제 적용되도록 합니다.
최종 평가·시각화의 기반 예측·속도 측정은 FP32를 쓰며, 학습 AMP는 기본 True입니다.

In [ ]:
MODEL_NAME = "yolo26m.pt"
LABEL_VERSION = "v1"
SEED = 42
FOLDS = tuple(range(N_FOLDS))
SPLIT_NAME = f"setup30fix_{LABEL_VERSION}_s{SEED}"
EPOCHS = 30
PATIENCE = 5  # 10으로 바꾸려면 여기만 수정한다.
LR0 = 0.001
OPTIMIZER = "MuSGD"
BATCH = 16
IMGSZ = 640
FREEZE = None
AUG = "on"
EVAL_CONF = 0.001
CM_CONFS = (0.001, 0.25)
VIS_CONF = 0.25
CM_MATCH_IOU = 0.45  # 라이브러리 기본 정답 매칭 기준. NMS IoU와 구분한다.
NMS_IOU = 0.5
NMS_ENABLED = True
MAX_DET = 4
GPU_INDEX = 0
WORKERS = 0  # 노트북·Windows의 다중 프로세스 문제를 피한다.
VERIFY_VALIDATION_WEIGHTS = True
BENCHMARK_WARMUP = 5
BENCHMARK_ROUNDS = 3

run_stamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f")
EXPERIMENT_DIR = RUNS_DIR / f"baseline_yolo26m_5fold_s{SEED}_{run_stamp}"
PREPARED_DIR = EXPERIMENT_DIR / "prepared"
PRETRAINED_PATH = EXPERIMENT_DIR / "pretrained" / MODEL_NAME

EXPERIMENT_CONFIG = {
    "model": MODEL_NAME,
    "label": LABEL_VERSION,
    "split": SPLIT_NAME,
    "folds": list(FOLDS),
    "seed": SEED,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "optimizer": OPTIMIZER,
    "lr0": LR0,
    "batch": BATCH,
    "imgsz": IMGSZ,
    "transfer": "full fine tuning",
    "freeze": FREEZE,
    "augmentation": AUG_PRESETS[AUG],
    "main_metric": MAIN_METRIC,
    "eval_conf": EVAL_CONF,
    "cm_confs": list(CM_CONFS),
    "vis_conf": VIS_CONF,
    "cm_match_iou": CM_MATCH_IOU,
    "nms": NMS_ENABLED,
    "nms_iou": NMS_IOU,
    "agnostic_nms": False,
    "max_det": MAX_DET,
    "rare_max": RARE_MAX,
    "setup_distance_px": 30,
    "setup_min_common_pills": MIN_SAME,
    "workers": WORKERS,
    "verify_validation_weights": VERIFY_VALIDATION_WEIGHTS,
    "evaluation_precision": "FP32",
    "benchmark_batch": 1,
    "benchmark_warmup": BENCHMARK_WARMUP,
    "benchmark_rounds": BENCHMARK_ROUNDS,
}
display(pd.Series(EXPERIMENT_CONFIG, name="실험 설정").to_frame())

In [ ]:
# GPU가 없으면 CPU로 자동 대체하지 않고 중단한다.
selected_device = get_device()
if not torch.cuda.is_available() or torch.device(selected_device).type != "cuda":
    raise RuntimeError(
        "로컬 NVIDIA GPU를 사용할 수 없습니다. 프로젝트 uv 커널, "
        "CUDA 지원 PyTorch와 NVIDIA 드라이버를 확인하세요."
    )
if not 0 <= GPU_INDEX < torch.cuda.device_count():
    raise ValueError(f"GPU_INDEX={GPU_INDEX}, GPU 수={torch.cuda.device_count()}")
if not {"nms", "quantize"} <= DEFAULT_CFG_DICT.keys():
    raise RuntimeError("프로젝트의 Ultralytics 8.4.182 이상 환경을 선택하세요.")
if len(FOLDS) != 5 or LABEL_VERSION != "v1" or VIS_CONF not in CM_CONFS:
    raise ValueError("이 베이스라인은 v1, 전체 5-fold, 혼동행렬과 같은 표시 신뢰도를 사용합니다.")
torch.cuda.set_device(GPU_INDEX)
DEVICE = f"{selected_device}:{GPU_INDEX}"
init_seeds(SEED, deterministic=True)

gpu = torch.cuda.get_device_properties(GPU_INDEX)
ENVIRONMENT = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda": torch.version.cuda,
    "device": DEVICE,
    "gpu": gpu.name,
    "gpu_memory_gib": round(gpu.total_memory / 1024**3, 2),
}
raw_root = PROJECT_ROOT / "data" / "raw"
RAW_DIR = raw_root / "sprint_ai_project1_data"
if not RAW_DIR.is_dir():
    RAW_DIR = raw_root
TRAIN_IMAGES = list_images(RAW_DIR / "train_images")
TEST_IMAGES = list_images(RAW_DIR / "test_images")
if (len(TRAIN_IMAGES), len(TEST_IMAGES)) != (232, 842):
    raise ValueError(
        f"확인한 원본 수와 다릅니다: train={len(TRAIN_IMAGES)}, test={len(TEST_IMAGES)}"
    )
if len({path.name for path in TRAIN_IMAGES}) != len(TRAIN_IMAGES):
    raise ValueError("중복된 train 이미지 파일명이 있습니다.")
display(pd.Series(ENVIRONMENT, name="GPU 환경").to_frame())
print(f"결과 폴더: {EXPERIMENT_DIR}")

## 2. EDA 보정 라벨과 희귀 묶음 고정 5-fold

현재 v1 파이프라인은 누락 8개 추가, kwon 좌표 수정, 검수된 19232 캡슐 보정을 사용합니다.
EDA 노트북의 과거 미결정 설명과 현재 파이프라인의 적용 상태는 다릅니다.
이 노트북은 현재 LABEL_VERSIONS["v1"]을 따르며 새 검수 승인을 하지 않습니다.
여백 보정(v2)은 적용하지 않습니다. 보정 후 라벨로 희귀 클래스와 세팅 묶음을 계산합니다.

희귀 약의 묶음 전체를 모든 fold의 train에 고정합니다. 나머지를 5개로 나눠 순환 검증합니다.
고정 묶음은 validation에 한 번도 나오지 않고, 나머지 이미지는 정확히 한 번 나와야 합니다.
같은 조합·세팅 누수 0, 희귀 클래스 validation 포함 0을 확인합니다.

In [ ]:
def write_json(path: Path, content: dict | list) -> None:
    """실험 기록을 UTF-8 JSON으로 저장한다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")


def prepare_data() -> list[Path]:
    """공용 데이터는 보존하고 실험 폴더 안에서 기존 5-fold 파이프라인을 실행한다."""
    import matplotlib

    previous_backend = matplotlib.get_backend()
    try:
        from data_process import data_main
    finally:
        plt.switch_backend(previous_backend)
    if PREPARED_DIR.exists():
        raise FileExistsError(
            "이미 준비한 실험입니다. 설정 셀에서 새 실험을 만들거나 다음 셀로 진행하세요."
        )
    source_dir = PROJECT_ROOT / "data" / "processed"
    required = ("added_annotations.csv", "fixed_annotations.csv")
    for name in required:
        if not (source_dir / name).is_file():
            raise FileNotFoundError(f"기존 보정 기록이 없습니다: {source_dir / name}")
    PREPARED_DIR.mkdir(parents=True)
    for name in (*required, "padding_review.csv"):
        if (source_dir / name).is_file():
            shutil.copy2(source_dir / name, PREPARED_DIR / name)
    # 기존 소스는 수정하지 않고, run()이 쓰는 출력 경로만 호출 동안 연결한다.
    with (
        patch.object(data_main, "PROCESSED_DIR", PREPARED_DIR),
        patch.object(data_yaml, "YOLO_ROOT", PREPARED_DIR / "yolo"),
    ):
        paths = data_main.run(
            LABEL_VERSION, [SPLIT_NAME], use_code_names=True, n_check=0, seed=SEED
        )
        expected = [data_yaml.data_yaml_path(LABEL_VERSION, SPLIT_NAME, fold) for fold in FOLDS]
        if paths != expected:
            raise AssertionError(f"5-fold data.yaml 경로가 다릅니다: {paths}")
    return expected


def audit_folds(labels: pd.DataFrame, folds: pd.DataFrame) -> pd.DataFrame:
    """5개 분할의 묶음 누수·희귀 약 고정·검증 방문 횟수를 검사한다."""
    rare_ids = set(rare_classes(labels))
    expected_files = set(labels["file_name"])
    if set(folds["fold"].unique()) != set(FOLDS):
        raise AssertionError("전체 5-fold가 필요합니다.")
    if folds.duplicated(["fold", "file_name"]).any():
        raise AssertionError("같은 fold에 중복 이미지가 있습니다.")
    rows = []
    fixed_files = None
    for fold, table in folds.groupby("fold", sort=True):
        if set(table["file_name"]) != expected_files:
            raise AssertionError(f"fold {fold}: 이미지 누락")
        if not set(table["split"]) == {"train", "val"}:
            raise AssertionError(f"fold {fold}: train/val이 모두 필요합니다.")
        for group_key in ("combo", "group"):
            if table.groupby(group_key)["split"].nunique().gt(1).any():
                raise AssertionError(f"fold {fold}: {group_key} 누수")
        mapping = table.set_index("file_name")
        box_groups = labels["file_name"].map(mapping["group"])
        rare_groups = set(box_groups[labels["category_id"].isin(rare_ids)])
        should_fix = table["group"].isin(rare_groups)
        if not table["fixed"].eq(should_fix).all():
            raise AssertionError(f"fold {fold}: 희귀 약 묶음 전체가 고정되지 않았습니다.")
        if not table.loc[should_fix, "split"].eq("train").all():
            raise AssertionError(f"fold {fold}: 고정 묶음이 validation에 있습니다.")
        fixed_now = set(table.loc[should_fix, "file_name"])
        if fixed_files is not None and fixed_now != fixed_files:
            raise AssertionError("fold마다 train 고정 이미지가 다릅니다.")
        fixed_files = fixed_now
        train_files = set(table.loc[table["split"] == "train", "file_name"])
        val_files = set(table.loc[table["split"] == "val", "file_name"])
        train_ids = set(labels.loc[labels["file_name"].isin(train_files), "category_id"])
        val_ids = set(labels.loc[labels["file_name"].isin(val_files), "category_id"])
        if rare_ids & val_ids:
            raise AssertionError(f"fold {fold}: 희귀 클래스가 validation에 있습니다.")
        rows.append(
            {
                "fold": int(fold),
                "train_images": len(train_files),
                "val_images": len(val_files),
                "fixed_train_images": len(fixed_now),
                "val_ratio": len(val_files) / len(table),
                "val_only_classes": sorted(int(value) for value in val_ids - train_ids),
                "classes_not_in_val": len(set(labels["category_id"]) - val_ids),
                "combo_leak": 0,
                "setup_leak": 0,
                "rare_in_val": 0,
            }
        )
    visits = folds.loc[folds["split"] == "val", "file_name"].value_counts()
    for name in expected_files:
        if visits.get(name, 0) != (0 if name in fixed_files else 1):
            raise AssertionError(f"검증 방문 횟수가 잘못됐습니다: {name}")
    return pd.DataFrame(rows)

In [ ]:
# 실행 시 데이터 준비 파일만 생성한다. 학습은 4절의 학습 셀에서 시작한다.
DATA_YAMLS = dict(zip(FOLDS, prepare_data(), strict=True))
LABELS = pd.read_csv(PREPARED_DIR / "labels" / f"labels_{LABEL_VERSION}.csv")
CLASS_MAP = pd.read_csv(PREPARED_DIR / "yolo" / LABEL_VERSION / "class_map.csv")
FOLD_TABLE = pd.concat(
    [pd.read_csv(PREPARED_DIR / "splits" / SPLIT_NAME / f"fold_{fold}.csv") for fold in FOLDS],
    ignore_index=True,
)
SPLIT_META = json.loads(
    (PREPARED_DIR / "splits" / SPLIT_NAME / "meta.json").read_text(encoding="utf-8")
)
if set(LABELS["file_name"]) != {path.name for path in TRAIN_IMAGES}:
    raise AssertionError("v1 라벨과 원본 train 이미지가 다릅니다.")
FOLD_AUDIT = audit_folds(LABELS, FOLD_TABLE)
display(FOLD_AUDIT)
print(f"적용 라벨: {LABEL_VERSION}, bbox={len(LABELS)}, 클래스={len(CLASS_MAP)}")
print(f"희귀 클래스(bbox {RARE_MAX}개 이하): {rare_classes(LABELS)}")
print(
    f"전 fold train 고정: {SPLIT_META['fixed_train_images']}장 / "
    f"{SPLIT_META['fixed_train_groups']}묶음"
)
print(
    f"실제 이미지당 최대 정답 수: {LABELS.groupby('file_name').size().max()} / 탐지 제한: {MAX_DET}"
)
print("train 전용 희귀 클래스의 validation AP는 평가 불가(NaN)로 기록합니다.")
FOLD_AUDIT.to_csv(EXPERIMENT_DIR / "fold_audit.csv", index=False)
write_json(
    EXPERIMENT_DIR / "experiment.json",
    {
        "config": EXPERIMENT_CONFIG,
        "environment": ENVIRONMENT,
        "split": SPLIT_META,
        "label_corrections": list(LABEL_VERSIONS[LABEL_VERSION]),
        "source_correction_sha256": {
            path.name: file_sha256(path) for path in PREPARED_DIR.glob("*.csv")
        },
    },
)

## 3. 검증 보호·주 지표·조기 중단

mAP75–95는 IoU 0.75, 0.80, 0.85, 0.90, 0.95의 클래스별 AP 평균입니다.
기존 Map7595Trainer로 best.pt와 early stopping 기준을 일치시킵니다.
0점이 이어져도 5에폭간 개선이 없으면 중단하도록 stopper를 연결합니다.

혼동행렬의 conf는 명시적으로 적용합니다. 정답 매칭 IoU=0.45는 라이브러리 기본값이며
NMS IoU=0.5와 별도로 기록합니다. Ultralytics Precision/Recall 요약은 IoU=0.5의
max-F1 신뢰도 지점 macro 평균입니다. 고정 conf에서의 혼동행렬 micro P/R은 별도 열입니다.

In [ ]:
class ExactConfusionMatrix(ConfusionMatrix):
    """요청한 신뢰도로 먼저 걸러 혼동행렬의 임계값을 명확하게 적용한다."""

    def process_batch(
        self,
        detections: dict[str, torch.Tensor],
        batch: dict,
        conf: float = 0.25,
        iou_thres: float = CM_MATCH_IOU,
    ) -> None:
        mask = detections["conf"] > conf
        filtered = {key: value[mask] for key, value in detections.items()}
        super().process_batch(filtered, batch, conf=0.0, iou_thres=iou_thres)


class ReadOnlyValidator(DetectionValidator):
    """검증에서 가중치가 바뀌지 않는지 확인하고 평가에 쓴 예측 박스를 보관한다."""

    @torch.inference_mode()
    def __call__(self, trainer=None, model=None, **kwargs) -> dict | None:
        protected = []
        if VERIFY_VALIDATION_WEIGHTS:
            candidates = (
                [trainer.model, trainer.ema.ema]
                if trainer is not None
                else [model]
                if isinstance(model, torch.nn.Module)
                else []
            )
            for candidate in candidates:
                snapshot = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in candidate.state_dict().items()
                }
                protected.append((candidate, snapshot))
        stats = super().__call__(trainer=trainer, model=model, **kwargs)
        for candidate, snapshot in protected:
            current = candidate.state_dict()
            changed = [
                name
                for name, tensor in snapshot.items()
                if name not in current or not torch.equal(tensor, current[name].detach().cpu())
            ]
            if changed or snapshot.keys() != current.keys():
                raise AssertionError(f"검증 중 가중치/버퍼가 바뀌었습니다: {changed[:5]}")
        return stats

    def init_metrics(self, model: torch.nn.Module) -> None:
        if torch.is_grad_enabled() or model.training:
            raise AssertionError("검증은 gradient 비활성화 + eval 모드여야 합니다.")
        super().init_metrics(model)
        if self.end2end or not self.args.nms or self.args.agnostic_nms:
            raise AssertionError("클래스별 NMS 경로가 필요합니다.")
        if self.args.max_det != MAX_DET or self.args.iou != NMS_IOU:
            raise AssertionError("합의한 NMS IoU/max_det와 다릅니다.")
        self.confusion_matrix = ExactConfusionMatrix(names=self.names)
        self.validation_predictions = []

    def postprocess(self, preds: torch.Tensor) -> list[dict[str, torch.Tensor]]:
        outputs = super().postprocess(preds)
        if any(len(output["conf"]) > MAX_DET for output in outputs):
            raise AssertionError("max_det 제한이 적용되지 않았습니다.")
        return outputs

    def update_metrics(self, preds: list[dict], batch: dict) -> None:
        super().update_metrics(preds, batch)
        if self.training:
            return
        # 별도 predict() 대신 평가한 박스를 원본 이미지 좌표로 되돌려 시각화에 사용한다.
        for index, prediction in enumerate(preds):
            prepared = self._prepare_batch(index, batch)
            original = self.scale_preds(prediction, prepared)
            self.validation_predictions.append(
                {
                    "file_name": Path(prepared["im_file"]).name,
                    "xyxy": original["bboxes"].detach().cpu().tolist(),
                    "yolo_idx": original["cls"].int().cpu().tolist(),
                    "conf": original["conf"].detach().cpu().tolist(),
                }
            )

    def finalize_metrics(self) -> None:
        super().finalize_metrics()
        self.metrics.validation_predictions = self.validation_predictions


class StrictEarlyStopping(EarlyStopping):
    """첫 평가 이후에는 엄격한 점수 증가만 개선으로 인정한다."""

    def __call__(self, epoch: int, fitness: float | None) -> bool:
        if fitness is None:
            return False
        if not np.isfinite(fitness):
            raise ValueError(f"유효하지 않은 검증 점수: {fitness}")
        if self.best_epoch == 0 or fitness > self.best_fitness:
            self.best_epoch, self.best_fitness = epoch, fitness
        delta = epoch - self.best_epoch
        self.possible_stop = delta >= self.patience - 1
        stop = delta >= self.patience
        if stop:
            print(f"Early stopping: {delta}에폭 개선 없음, best epoch={self.best_epoch}")
        return stop


class BaselineTrainer(Map7595Trainer):
    """기존 mAP75–95 학습기에 검증 보호를 연결한다."""

    def get_validator(self) -> ReadOnlyValidator:
        return ReadOnlyValidator(
            self.test_loader,
            save_dir=self.save_dir,
            args=copy(self.args),
            _callbacks=self.callbacks,
        )


def start_training(trainer: BaselineTrainer) -> None:
    """엄격한 stopper를 적용하고 실제 옵티마이저·학습률 그룹을 남긴다."""
    trainer.stopper = StrictEarlyStopping(patience=trainer.args.patience)
    info = {
        "optimizer": type(trainer.optimizer).__name__,
        "lr0": trainer.args.lr0,
        "group_initial_lrs": [
            float(group.get("initial_lr", group["lr"])) for group in trainer.optimizer.param_groups
        ],
        "trainable_parameters": sum(
            parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad
        ),
        "frozen_parameters": sum(
            parameter.numel()
            for parameter in trainer.model.parameters()
            if not parameter.requires_grad
        ),
    }
    write_json(Path(trainer.save_dir) / "optimizer_used.json", info)
    print(json.dumps(info, ensure_ascii=False))


def log_epoch(trainer: BaselineTrainer) -> None:
    """매 에폭 train/val 손실과 검증 지표를 함께 출력한다."""
    if not trainer.validator.training:
        return
    values = {
        "epoch": trainer.epoch + 1,
        **trainer.label_loss_items(trainer.tloss, prefix="train"),
        **{key: float(value) for key, value in trainer.metrics.items()},
    }
    print(json.dumps(values, ensure_ascii=False))

## 4. 5-fold 학습

이 셀에서 실제 학습이 시작됩니다. fold마다 동일한 사전학습 가중치를 새로 불러옵니다.
완료한 fold는 동일 실험 설정을 확인한 후 건너뜁니다. 미완료 폴더는 자동 덮어쓰지 않습니다.
OOM이 발생해도 batch=16을 임의로 줄이지 않습니다.

각 fold의 best.pt는 해당 fold의 검증 mAP75–95로 선택하며, 하나의 최고 fold만 최종 점수로 쓰지 않습니다.

In [ ]:
def train_fold(fold: int) -> Path:
    """한 fold를 사전학습 가중치부터 학습하고 best.pt 경로를 반환한다."""
    train_dir = EXPERIMENT_DIR / f"fold_{fold}" / "train"
    complete_path = train_dir / "completed.json"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text(encoding="utf-8"))
        best = Path(complete["best_pt"])
        if complete["config"] != EXPERIMENT_CONFIG or not best.is_file():
            raise ValueError(f"fold {fold}: 기존 완료 기록과 설정/가중치가 다릅니다.")
        print(f"fold {fold}: 완료된 학습 사용: {best}")
        return best
    if train_dir.exists():
        raise FileExistsError(
            f"미완료 결과가 있습니다: {train_dir}. 새 실험 설정 셀부터 실행하세요."
        )
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAINED_PATH.is_file() and (PROJECT_ROOT / MODEL_NAME).is_file():
        shutil.copy2(PROJECT_ROOT / MODEL_NAME, PRETRAINED_PATH)
    init_seeds(SEED, deterministic=True)
    model = YOLO(str(PRETRAINED_PATH))
    model.to(DEVICE)
    model.add_callback("on_train_start", start_training)
    model.add_callback("on_fit_epoch_end", log_epoch)
    args = {
        **AUG_PRESETS[AUG],
        "data": str(DATA_YAMLS[fold]),
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "seed": SEED,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "freeze": FREEZE,
        "device": DEVICE,
        "workers": WORKERS,
        "deterministic": True,
        "val": True,
        "save": True,
        "plots": True,
        "amp": True,
        "fraction": 1.0,
        "cos_lr": False,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "max_det": MAX_DET,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "project": str(train_dir.parent),
        "name": train_dir.name,
        "exist_ok": False,
    }
    try:
        model.train(trainer=BaselineTrainer, **args)
        best = Path(model.trainer.best)
        if not best.is_file():
            raise FileNotFoundError(f"best.pt가 없습니다: {best}")
        write_json(
            complete_path,
            {
                "best_pt": str(best),
                "config": EXPERIMENT_CONFIG,
                "pretrained_sha256": file_sha256(PRETRAINED_PATH),
            },
        )
        return best
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


BEST_WEIGHTS = {}
for fold in FOLDS:
    print(f"===== fold {fold + 1}/{len(FOLDS)} =====")
    BEST_WEIGHTS[fold] = train_fold(fold)
display(pd.Series({fold: str(path) for fold, path in BEST_WEIGHTS.items()}, name="best.pt"))

In [ ]:
# 모든 fold의 실제 수행 에폭까지 train/val 손실과 검증 지표를 표시한다.
for fold in FOLDS:
    history = pd.read_csv(EXPERIMENT_DIR / f"fold_{fold}" / "train" / "results.csv")
    history.columns = history.columns.str.strip()
    losses = [
        column
        for column in history
        if column.startswith("train/") and f"val/{column.split('/', 1)[1]}" in history
    ]
    fig, axes = plt.subplots(len(losses) + 1, 1, figsize=(11, 3 * (len(losses) + 1)), squeeze=False)
    for axis, train_key in zip(axes[:, 0], losses, strict=False):
        val_key = f"val/{train_key.split('/', 1)[1]}"
        axis.plot(history["epoch"], history[train_key], label=train_key)
        axis.plot(history["epoch"], history[val_key], label=val_key)
        axis.set(xlabel="Epoch", ylabel="Loss")
        axis.legend()
    for key in (MAIN_METRIC_KEY, "metrics/precision(B)", "metrics/recall(B)"):
        if key in history:
            axes[-1, 0].plot(history["epoch"], history[key], label=key)
    axes[-1, 0].set(xlabel="Epoch", ylabel="Validation metric", ylim=(0, 1))
    axes[-1, 0].legend()
    fig.suptitle(f"Fold {fold} | Train / Validation")
    fig.tight_layout()
    fig.savefig(EXPERIMENT_DIR / f"fold_{fold}" / "learning_curves.png", dpi=150)
    plt.show()
    plt.close(fig)

## 5. best.pt 평가 · 두 신뢰도의 혼동행렬 · 클래스별 AP

각 fold에서 같은 best.pt와 validation을 사용하고 conf만 0.001/0.25로 바꿉니다.
총 10개 평가의 원시·정규화 혼동행렬을 저장하고 같은 색상 범위로 비교합니다.
주 성능은 conf=0.001입니다. 검증에 없는 클래스 AP는 0으로 채우지 않고 NaN으로 둡니다.

In [ ]:
def per_class_metrics(metrics: object, fold: int) -> pd.DataFrame:
    """검증에 없는 클래스는 NaN으로 두고 클래스별 AP·학습/검증 수를 만든다."""
    table = CLASS_MAP.copy()
    table["group"] = table["category_id"].map(LABELS["category_id"].value_counts().map(size_group))
    mapping = FOLD_TABLE.loc[FOLD_TABLE["fold"] == fold].set_index("file_name")["split"]
    box_split = LABELS["file_name"].map(mapping)
    for split in ("train", "val"):
        counts = LABELS.loc[box_split == split, "category_id"].value_counts()
        table[f"n_{split}_bbox"] = table["category_id"].map(counts).fillna(0).astype(int)
    box = metrics.box
    scores = pd.DataFrame(
        {
            "ap50": box.ap50,
            "ap50_95": box.ap,
            "ap75_95": ap75_95(box.all_ap) if len(box.all_ap) else [],
            "precision_max_f1": box.p,
            "recall_max_f1": box.r,
        },
        index=box.ap_class_index,
    )
    table = table.join(scores, on="yolo_idx")
    present = table["n_val_bbox"] > 0
    table.loc[present, scores.columns] = table.loc[present, scores.columns].fillna(0.0)
    table.loc[table["n_val_bbox"] == 0, scores.columns] = np.nan
    table.insert(0, "fold", fold)
    return table


def compare_confusion_matrices(matrices: dict[float, np.ndarray], output_path: Path) -> None:
    """동일한 색상 범위로 두 혼동행렬을 나란히 저장한다."""
    names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    maximum = max(1.0, max(float(matrix.max()) for matrix in matrices.values()))
    fig, axes = plt.subplots(1, len(CM_CONFS), figsize=(24, 11), squeeze=False)
    for axis, confidence in zip(axes[0], CM_CONFS, strict=True):
        mesh = axis.imshow(matrices[confidence], cmap="Blues", vmin=0, vmax=maximum)
        axis.set(
            title=f"conf={confidence:g}, match IoU={CM_MATCH_IOU}",
            xlabel="True",
            ylabel="Predicted",
            xticks=range(len(names)),
            yticks=range(len(names)),
            xticklabels=names,
            yticklabels=names,
        )
        axis.tick_params(axis="x", labelrotation=90, labelsize=6)
        axis.tick_params(axis="y", labelsize=6)
    fig.colorbar(mesh, ax=axes.ravel().tolist(), label="Count", fraction=0.02, pad=0.02)
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.close(fig)


def evaluate_fold_baseline(fold: int, weights: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """한 fold를 두 신뢰도로 평가하고 지표·행렬·실제 평가 예측을 저장한다."""
    out_dir = EXPERIMENT_DIR / f"fold_{fold}" / "evaluation"
    matrices, rows = {}, []
    main_per_class = None
    class_names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    class_index = dict(zip(CLASS_MAP["category_id"], CLASS_MAP["yolo_idx"], strict=True))
    expected_files = set(
        FOLD_TABLE.loc[(FOLD_TABLE["fold"] == fold) & (FOLD_TABLE["split"] == "val"), "file_name"]
    )
    for confidence in CM_CONFS:
        model = YOLO(str(weights))
        try:
            metrics = model.val(
                validator=ReadOnlyValidator,
                data=str(DATA_YAMLS[fold]),
                split="val",
                imgsz=IMGSZ,
                batch=BATCH,
                device=DEVICE,
                workers=WORKERS,
                conf=confidence,
                iou=NMS_IOU,
                max_det=MAX_DET,
                nms=NMS_ENABLED,
                agnostic_nms=False,
                quantize=32,
                plots=True,
                augment=False,
                save_json=False,
                project=str(out_dir),
                name=f"conf_{confidence:g}",
                exist_ok=True,
            )
            predictions = metrics.validation_predictions
            if (
                len(predictions) != len(expected_files)
                or {record["file_name"] for record in predictions} != expected_files
            ):
                raise AssertionError("평가 예측 이미지와 해당 fold의 validation이 다릅니다.")
            write_json(out_dir / f"predictions_conf_{confidence:g}.json", predictions)
            matrix = metrics.confusion_matrix.matrix.copy()
            matrices[confidence] = matrix
            pd.DataFrame(matrix.astype(int), index=class_names, columns=class_names).to_csv(
                out_dir / f"confusion_matrix_conf_{confidence:g}.csv", index_label="predicted/true"
            )
            true_positives = float(np.trace(matrix[:-1, :-1]))
            predicted = float(matrix[:-1, :].sum())
            targets = float(matrix[:, :-1].sum())
            table = per_class_metrics(metrics, fold)
            if confidence == EVAL_CONF:
                main_per_class = table
                table.to_csv(out_dir / "per_class.csv", index=False)
            rows.append(
                {
                    "fold": fold,
                    "conf": confidence,
                    MAIN_METRIC: map75_95(metrics.box),
                    "precision_macro_max_f1_iou0.5": float(metrics.box.mp),
                    "recall_macro_max_f1_iou0.5": float(metrics.box.mr),
                    "precision_micro_fixed_conf_cm": true_positives / predicted
                    if predicted
                    else 0.0,
                    "recall_micro_fixed_conf_cm": true_positives / targets if targets else 0.0,
                    "cm_match_iou": CM_MATCH_IOU,
                    "n_predictions": int(predicted),
                    "n_ground_truths": int(targets),
                    "classes_not_in_val": int(table["n_val_bbox"].eq(0).sum()),
                    "val_only_classes": int(
                        ((table["n_train_bbox"] == 0) & (table["n_val_bbox"] > 0)).sum()
                    ),
                    **{
                        f"ap75_95_group_{group}": float(
                            table.loc[table["group"] == group, "ap75_95"].mean()
                        )
                        for group in ("21+", "6-20", "≤5")
                    },
                    **{
                        f"confuse_{first}_{second}": int(
                            matrix[class_index[first], class_index[second]]
                            + matrix[class_index[second], class_index[first]]
                        )
                        for first, second in CONFUSION_PAIRS
                        if first in class_index and second in class_index
                    },
                    "validation_batch": BATCH,
                    "val_forward_ms_per_image": float(metrics.speed["inference"]),
                    "nms_iou": NMS_IOU,
                    "max_det": MAX_DET,
                }
            )
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    comparison = out_dir / "confusion_matrix_comparison.png"
    compare_confusion_matrices(matrices, comparison)
    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "threshold_comparison.csv", index=False)
    display(result)
    display(Image(filename=str(comparison)))
    return result, main_per_class

In [ ]:
threshold_results, per_class_results = [], []
for fold in FOLDS:
    comparison, per_class = evaluate_fold_baseline(fold, BEST_WEIGHTS[fold])
    threshold_results.append(comparison)
    per_class_results.append(per_class)
    # 중간 결과도 fold마다 저장한다.
    pd.concat(threshold_results, ignore_index=True).to_csv(
        EXPERIMENT_DIR / "threshold_comparison_all_folds.csv", index=False
    )
THRESHOLD_RESULTS = pd.concat(threshold_results, ignore_index=True)
PER_CLASS_RESULTS = pd.concat(per_class_results, ignore_index=True)
PER_CLASS_RESULTS.to_csv(EXPERIMENT_DIR / "per_class_all_folds.csv", index=False)
PRIMARY_RESULTS = THRESHOLD_RESULTS.loc[THRESHOLD_RESULTS["conf"] == EVAL_CONF].copy()
metric_columns = [
    MAIN_METRIC,
    "precision_macro_max_f1_iou0.5",
    "recall_macro_max_f1_iou0.5",
    "precision_micro_fixed_conf_cm",
    "recall_micro_fixed_conf_cm",
    "ap75_95_group_21+",
    "ap75_95_group_6-20",
    "ap75_95_group_≤5",
]
SUMMARY = PRIMARY_RESULTS[metric_columns].agg(["mean", "std", "count"]).T
SUMMARY.to_csv(EXPERIMENT_DIR / "summary_5fold.csv")
display(PRIMARY_RESULTS)
display(SUMMARY)
class_summary = PER_CLASS_RESULTS.groupby(["category_id", "dl_name", "group"])["ap75_95"].agg(
    mean="mean", std="std", evaluated_folds="count"
)
class_summary.to_csv(EXPERIMENT_DIR / "per_class_summary_5fold.csv")
display(class_summary)
print("count=0인 희귀 클래스 AP는 평가 불가입니다. std는 표본 표준편차(ddof=1)입니다.")

## 6. Validation 정답 bbox · 예측 bbox 이미지

conf=0.25 평가에서 실제 사용한 예측을 가져옵니다. 평가를 다시 추론하지 않습니다.
**왼쪽: 정답(초록), 가운데: 예측(주황·점수 표시), 오른쪽: 겹쳐 보기**입니다.
정답은 보정된 v1 라벨, 예측은 원본 이미지 좌표입니다.

모든 fold의 모든 validation 이미지 비교본을 저장합니다. 아래 조회 셀에서 fold와 시작 위치를
바꿔 원하는 페이지를 볼 수 있습니다. 표시 신뢰도는 평가 conf=0.001과 별도인 VIS_CONF=0.25입니다.

In [ ]:
def draw_box(axis: object, xyxy: list[float], label: str, color: str, linestyle: str = "-") -> None:
    """원본 픽셀 좌표로 박스와 클래스·점수를 그린다."""
    x1, y1, x2, y2 = xyxy
    axis.add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor=color,
            linewidth=1.8,
            linestyle=linestyle,
        )
    )
    axis.text(
        x1,
        max(0, y1 - 5),
        label,
        color="white",
        fontsize=8,
        bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 2},
    )


def save_validation_images(fold: int) -> list[Path]:
    """해당 fold의 모든 검증 이미지에 정답·실제 평가 예측 박스를 그려 저장한다."""
    eval_dir = EXPERIMENT_DIR / f"fold_{fold}" / "evaluation"
    records = json.loads(
        (eval_dir / f"predictions_conf_{VIS_CONF:g}.json").read_text(encoding="utf-8")
    )
    expected_files = set(
        FOLD_TABLE.loc[(FOLD_TABLE["fold"] == fold) & (FOLD_TABLE["split"] == "val"), "file_name"]
    )
    if {record["file_name"] for record in records} != expected_files:
        raise AssertionError("시각화 대상이 validation 목록과 다릅니다.")
    images_by_name = {path.name: path for path in TRAIN_IMAGES}
    category_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    image_dir = EXPERIMENT_DIR / f"fold_{fold}" / "val_images"
    image_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for record in sorted(records, key=lambda item: item["file_name"]):
        file_name = record["file_name"]
        bgr = cv2.imread(str(images_by_name[file_name]))
        if bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {file_name}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 3, figsize=(15, 6))
        for axis, title in zip(
            axes, ("Ground truth", f"Prediction conf>{VIS_CONF:g}", "Overlay"), strict=True
        ):
            axis.imshow(rgb)
            axis.set_title(title)
            axis.axis("off")
        truth = LABELS.loc[LABELS["file_name"] == file_name]
        for box in truth.itertuples():
            coords = [box.x, box.y, box.x + box.w, box.y + box.h]
            for index in (0, 2):
                draw_box(axes[index], coords, f"GT {box.category_id}", "green")
        for coords, class_index, score in zip(
            record["xyxy"], record["yolo_idx"], record["conf"], strict=True
        ):
            category = category_by_index[class_index]
            for index in (1, 2):
                draw_box(axes[index], coords, f"Pred {category} {score:.3f}", "darkorange", "--")
        if not record["xyxy"]:
            axes[1].text(0.5, 0.05, "No predictions", transform=axes[1].transAxes, ha="center")
        fig.suptitle(f"Fold {fold} | {file_name}", fontsize=9)
        fig.tight_layout()
        path = image_dir / f"{Path(file_name).stem}.jpg"
        fig.savefig(path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        paths.append(path)
    return paths


def show_validation_images(fold: int = 0, start: int = 0, count: int = 4) -> None:
    """저장된 검증 비교 이미지를 지정한 페이지부터 노트북에 표시한다."""
    paths = sorted((EXPERIMENT_DIR / f"fold_{fold}" / "val_images").glob("*.jpg"))
    if not paths:
        raise FileNotFoundError("먼저 검증 비교 이미지 저장 셀을 실행하세요.")
    if start < 0 or count < 1 or start >= len(paths):
        raise ValueError(f"start는 0~{len(paths) - 1}, count는 1 이상이어야 합니다.")
    print(f"Fold {fold}: {start + 1}~{min(start + count, len(paths))} / 전체 {len(paths)}장")
    for path in paths[start : start + count]:
        display(Image(filename=str(path)))


for fold in FOLDS:
    paths = save_validation_images(fold)
    print(f"fold {fold}: GT/예측 비교 이미지 {len(paths)}장 저장")

In [ ]:
# fold=0~4, start=0, 4, 8, ...로 바꿔 모든 검증 이미지를 볼 수 있다.
show_validation_images(fold=0, start=0, count=4)

## 7. GPU 추론속도 · 최종 5-fold 요약

각 fold의 best.pt와 해당 validation 이미지로 측정합니다. **batch=1, FP32, 640×640,
conf=0.001, 클래스별 NMS IoU=0.5, max_det=4**를 동일하게 적용합니다.
warmup 5회 후 validation 전체를 3회 반복하며 매 측정 전후 GPU를 동기화합니다.

forward_ms는 모델 연산 시간입니다. pipeline_ms는 메모리에 읽은 이미지부터 전처리·GPU 전송·
forward·후처리·Python 호출까지의 시간이며 디스크 읽기는 제외합니다.
학습·검증 batch=16에서 나온 속도와 구분합니다. 동일 GPU에서 다른 작업 없이 비교하세요.

In [ ]:
def benchmark_fold(fold: int, weights: Path) -> dict:
    """warmup·GPU 동기화 후 batch=1의 모델/전체 추론 시간을 기록한다."""
    val_names = set(
        FOLD_TABLE.loc[(FOLD_TABLE["fold"] == fold) & (FOLD_TABLE["split"] == "val"), "file_name"]
    )
    loaded = []
    for path in sorted(TRAIN_IMAGES):
        if path.name in val_names:
            image = cv2.imread(str(path))
            if image is None:
                raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
            loaded.append((path.name, image))
    if not loaded or BENCHMARK_WARMUP < 1 or BENCHMARK_ROUNDS < 1:
        raise ValueError("검증 이미지, warmup, 반복 수는 1 이상이어야 합니다.")
    model = YOLO(str(weights))
    model.to(DEVICE)
    model.model.eval()
    predict_args = {
        "imgsz": IMGSZ,
        "batch": 1,
        "rect": False,
        "device": DEVICE,
        "quantize": 32,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "max_det": MAX_DET,
        "augment": False,
        "save": False,
        "verbose": False,
        "stream": False,
    }
    rows = []
    try:
        with torch.inference_mode():
            for _ in range(BENCHMARK_WARMUP):
                model.predict(source=loaded[0][1], **predict_args)
            torch.cuda.synchronize(DEVICE)
            for repetition in range(BENCHMARK_ROUNDS):
                for file_name, image in loaded:
                    torch.cuda.synchronize(DEVICE)
                    start = perf_counter()
                    prediction = model.predict(source=image, **predict_args)[0]
                    torch.cuda.synchronize(DEVICE)
                    elapsed = (perf_counter() - start) * 1000
                    if len(prediction.boxes) > MAX_DET:
                        raise AssertionError("속도 측정 예측의 max_det가 다릅니다.")
                    rows.append(
                        {
                            "fold": fold,
                            "round": repetition + 1,
                            "file_name": file_name,
                            "pipeline_ms": elapsed,
                            "forward_ms": float(prediction.speed["inference"]),
                            "preprocess_ms": float(prediction.speed["preprocess"]),
                            "postprocess_ms": float(prediction.speed["postprocess"]),
                        }
                    )
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    timings = pd.DataFrame(rows)
    summary = {
        "fold": fold,
        "n_images": len(loaded),
        "n_measurements": len(timings),
        "forward_mean_ms": float(timings["forward_ms"].mean()),
        "pipeline_mean_ms": float(timings["pipeline_ms"].mean()),
        "pipeline_median_ms": float(timings["pipeline_ms"].median()),
        "pipeline_p95_ms": float(timings["pipeline_ms"].quantile(0.95)),
        "forward_fps": 1000 / float(timings["forward_ms"].mean()),
        "pipeline_fps": 1000 / float(timings["pipeline_ms"].mean()),
    }
    out_dir = EXPERIMENT_DIR / f"fold_{fold}" / "evaluation"
    timings.to_csv(out_dir / "inference_timings.csv", index=False)
    write_json(
        out_dir / "inference_speed.json",
        {
            **summary,
            **ENVIRONMENT,
            "batch": 1,
            "imgsz": IMGSZ,
            "precision": "FP32",
            "conf": EVAL_CONF,
            "nms_iou": NMS_IOU,
            "max_det": MAX_DET,
            "nms": NMS_ENABLED,
            "agnostic_nms": False,
            "disk_io_included": False,
            "warmup": BENCHMARK_WARMUP,
            "rounds": BENCHMARK_ROUNDS,
        },
    )
    return summary


SPEED_RESULTS = pd.DataFrame([benchmark_fold(fold, BEST_WEIGHTS[fold]) for fold in FOLDS])
SPEED_RESULTS.to_csv(EXPERIMENT_DIR / "inference_speed_5fold.csv", index=False)
FINAL_RESULTS = PRIMARY_RESULTS.merge(SPEED_RESULTS, on="fold", validate="1:1")
FINAL_RESULTS.to_csv(EXPERIMENT_DIR / "results_5fold.csv", index=False)
speed_columns = ["forward_mean_ms", "pipeline_mean_ms", "forward_fps", "pipeline_fps"]
FINAL_SUMMARY = FINAL_RESULTS[metric_columns + speed_columns].agg(["mean", "std", "count"]).T
FINAL_SUMMARY.to_csv(EXPERIMENT_DIR / "summary_5fold.csv")
display(FINAL_RESULTS)
display(FINAL_SUMMARY)
print(f"전체 결과: {EXPERIMENT_DIR}")
print("가중치: fold_0~4/train/weights/best.pt")
print("혼동행렬: fold_0~4/evaluation/conf_0.001 및 conf_0.25")
print("검증 정답/예측 이미지: fold_0~4/val_images/")

EDA의 그림·캐시·보정 후보와 실제 적용 라벨을 구분합니다. train에만 있는 희귀 클래스의
AP는 검증할 수 없으며, 전체 56종의 일반화 성능을 측정했다고 해석하지 않습니다.
검증에 없는 클래스 수와 평가된 fold 수를 클래스별 표에 남깁니다.

참고: [Ultralytics 검증](https://docs.ultralytics.com/modes/val/),
[후처리·신뢰도·정밀도 설정](https://docs.ultralytics.com/usage/cfg/).
실행한 라이브러리 버전과 모든 명시 설정은 experiment.json 및 fold별 args.yaml에 기록됩니다.